# Task 2.1 — Dataset Selection and Setup

**Paper:** *Dual Coordinate Descent Methods for Logistic Regression and Maximum Entropy Models* (Yu et al., 2011)

---

## Dataset: Breast Cancer Wisconsin (Diagnostic)

**Source:** `sklearn.datasets.load_breast_cancer()` — the standard UCI Breast Cancer Wisconsin dataset included in scikit-learn.

**Justification:** The Breast Cancer dataset is a binary classification task (malignant vs. benign, 569 instances, 30 real-valued features) that is directly aligned with the logistic regression problem studied in the paper. The paper's CDdual method is designed for exactly this setting: binary labels ($y \in \{-1, +1\}$), continuous numeric features, and a moderate sample size that allows CPU-based experimentation. The dataset has 212 negative and 357 positive instances (mild class imbalance), which provides a realistic test of whether the dual optimum structure (Theorem 1: all $\alpha^* \in (0, C)$) holds empirically. Compared to the paper's datasets (e.g., rcv1 with 677k instances and 47k features), the breast cancer dataset is smaller, so per-iteration timings will differ, but the algorithmic behaviour — convergence, dual variable distribution, and relative performance — is directly comparable. A limitation is that the paper focuses on high-dimensional sparse NLP/text data where the $O(n)$ per-step advantage of CDdual is most pronounced; the breast cancer dataset is dense and lower-dimensional, so CDdual's advantage over primal methods is less dramatic here.

## Preprocessing Steps

In [1]:
# ── Reproducibility ──────────────────────────────────────────────────────────
import numpy as np
np.random.seed(42)

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd

# Load dataset
data = load_breast_cancer()
X, y = data.data, data.target

# Convert labels to {-1, +1} as required by the LR dual formulation [Eq.1]
y_lr = 2 * y - 1

print(f"Dataset: Breast Cancer Wisconsin (Diagnostic)")
print(f"Total samples: {X.shape[0]}")
print(f"Features:      {X.shape[1]}")
print(f"Classes:       {data.target_names} (mapped to -1, +1)")
print(f"Class counts:  {dict(zip(data.target_names, np.bincount(y)))}")

Dataset: Breast Cancer Wisconsin (Diagnostic)
Total samples: 569
Features:      30
Classes:       ['malignant' 'benign'] (mapped to -1, +1)
Class counts:  {np.str_('malignant'): np.int64(212), np.str_('benign'): np.int64(357)}


The labels are converted from {0, 1} to {-1, +1} as required by the LR dual formulation (Section 1, Eq. 1 uses $y_i \in \{1, -1\}$).

In [2]:
# Train/test split: 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X, y_lr, test_size=0.2, random_state=42)

# Standardise features (zero mean, unit variance)
# Rationale: The paper's datasets (document TF-IDF features) are naturally bounded;
# breast cancer features span very different scales, so standardisation is necessary
# for the regulariser (1/2 w^T w in Eq.1) to treat all features equitably.
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print(f"Training set: {X_train.shape}")
print(f"Test set:     {X_test.shape}")
print(f"Feature mean after scaling (should be ~0): {X_train.mean():.4f}")
print(f"Feature std  after scaling (should be ~1): {X_train.std():.4f}")

# Save for reproducibility
np.save('data/X_train.npy', X_train)
np.save('data/X_test.npy',  X_test)
np.save('data/y_train.npy', y_train)
np.save('data/y_test.npy',  y_test)
print("\nData saved to partB/data/")

Training set: (455, 30)
Test set:     (114, 30)
Feature mean after scaling (should be ~0): -0.0000
Feature std  after scaling (should be ~1): 1.0000

Data saved to partB/data/
